# 01 — Binance REST API Exploration
Phase 1: Understanding the data before building anything.

## 1. Setup

In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from src.utils.time_utils import to_ms
from src.params.constants import BINANCE_KLINES_URL
from src.utils.dataframe_utils import raw_to_dataframe

## 2. Fetch Helper Function

In [2]:
def get_binance_data(
    symbol: str = "BTCUSDT",
    interval: str = "1d",
    start_time: int | None = None,
    end_time: int | None = None,
    limit: int = 10,
):
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit,
    }
    if start_time is not None:
        params["startTime"] = start_time
    if end_time is not None:
        params["endTime"] = end_time

    response = requests.get(BINANCE_KLINES_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

## 3. Initial Fetch — Raw Response
Fetch 3 recent daily candles and inspect the raw structure Binance returns.

In [3]:
data = get_binance_data(symbol="BTCUSDT", interval="1d", limit=3)

print(f"Number of candles returned: {len(data)}")
print(f"Number of fields per candle: {len(data[0])}")
print(f"\nFirst candle raw:")
print(data[0])

Number of candles returned: 3
Number of fields per candle: 12

First candle raw:
[1778457600000, '82210.07000000', '82380.00000000', '80462.97000000', '81745.65000000', '12951.76056000', 1778543999999, '1053193670.87575560', 2576396, '6132.37349000', '498860919.49640370', '0']


## 4. Convert to DataFrame
Assign column names, convert timestamps and numeric types.

In [4]:
df_btc = raw_to_dataframe(data)
print(df_btc.shape)
df_btc.head()

(3, 12)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2026-05-11,82210.07,82380.00,80462.97,81745.65,12951.76056,2026-05-11 23:59:59.999,1.053194e+09,2576396,6132.37349,4.988609e+08,0
1,2026-05-12,81745.66,81788.00,79843.59,80504.47,12386.25676,2026-05-12 23:59:59.999,1.000180e+09,2345137,5716.20385,4.617398e+08,0
2,2026-05-13,80504.47,81324.64,79606.00,79618.53,6944.32123,2026-05-13 23:59:59.999,5.606867e+08,1184372,3311.90625,2.674726e+08,0


In [5]:
print(df_btc.dtypes)

open_time                 datetime64[ms]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[ms]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
ignore                               str
dtype: object


## 5. How Far Back Does History Go?
Fetch from a very early date to find the first ever available candle.

In [6]:
earliest_data = get_binance_data(
    symbol="BTCUSDT",
    interval="1d",
    start_time=to_ms("01.01.2010T00:00"),
    limit=1
)

first_candle = pd.to_datetime(earliest_data[0][0], unit="ms")
print(f"Earliest available daily candle: {first_candle}")

Earliest available daily candle: 2017-08-17 00:00:00


## 6. Max Limit Per Request
What is the maximum number of candles Binance returns in one API call?

In [7]:
max_test = get_binance_data(symbol="BTCUSDT", interval="1d", limit=1500)
print(f"Requested: 1500 candles")
print(f"Returned:  {len(max_test)} candles")
print(f"=> Max limit per request is: {len(max_test)}")

Requested: 1500 candles
Returned:  1000 candles
=> Max limit per request is: 1000


## 7. Available Intervals
Fetch the same period with different intervals to understand granularity options.

In [8]:
intervals = ["1d", "1h", "15m", "5m", "1m"]

for interval in intervals:
    result = get_binance_data(
        symbol="BTCUSDT",
        interval=interval,
        start_time=to_ms("01.05.2026T00:00"),
        end_time=to_ms("08.05.2026T00:00"),
        limit=1500
    )
    print(f"interval={interval:>4}  =>  {len(result):>5} candles returned")

interval=  1d  =>      8 candles returned
interval=  1h  =>    169 candles returned
interval= 15m  =>    673 candles returned
interval=  5m  =>   1000 candles returned
interval=  1m  =>   1000 candles returned


## 8. Field Analysis — What to Keep for ML?
Review each of the 12 columns and decide: keep, redundant, or drop.

In [9]:
field_notes = {
    "open_time":               "KEEP  — primary key with symbol, used for ordering and joins",
    "open":                    "KEEP  — ML feature",
    "high":                    "KEEP  — ML feature, used for volatility",
    "low":                     "KEEP  — ML feature, used for volatility",
    "close":                   "KEEP  — ML feature, used for price change label",
    "volume":                  "KEEP  — ML feature, market activity signal",
    "close_time":              "KEEP  — useful for verifying no gaps between candles",
    "quote_asset_volume":      "DROP  — volume in USD terms, redundant when we have volume + close",
    "number_of_trades":        "KEEP  — ML feature, activity signal independent of volume",
    "taker_buy_base_volume":   "DROP  — advanced order flow, not needed for basic ML model",
    "taker_buy_quote_volume":  "DROP  — redundant with above",
    "ignore":                  "DROP  — reserved by Binance, always 0",
}

for col, note in field_notes.items():
    print(f"{col:<30} {note}")

open_time                      KEEP  — primary key with symbol, used for ordering and joins
open                           KEEP  — ML feature
high                           KEEP  — ML feature, used for volatility
low                            KEEP  — ML feature, used for volatility
close                          KEEP  — ML feature, used for price change label
volume                         KEEP  — ML feature, market activity signal
close_time                     KEEP  — useful for verifying no gaps between candles
quote_asset_volume             DROP  — volume in USD terms, redundant when we have volume + close
number_of_trades               KEEP  — ML feature, activity signal independent of volume
taker_buy_base_volume          DROP  — advanced order flow, not needed for basic ML model
taker_buy_quote_volume         DROP  — redundant with above
ignore                         DROP  — reserved by Binance, always 0


## 9. Multiple Symbols — Does ETHUSDT Have the Same Structure?

In [10]:
eth_data = get_binance_data(symbol="ETHUSDT", interval="1d", limit=3)
df_eth = raw_to_dataframe(eth_data)

print(f"ETHUSDT shape: {df_eth.shape}")
print(f"Same columns as BTCUSD T: {list(df_eth.columns) == list(df_btc.columns)}")
df_eth.head()

ETHUSDT shape: (3, 12)
Same columns as BTCUSD T: True


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2026-05-11,2371.26,2374.92,2304.00,2339.87,236962.0552,2026-05-11 23:59:59.999,5.534281e+08,2594593,103490.2032,2.417382e+08,0
1,2026-05-12,2339.88,2340.85,2256.27,2274.96,252981.1082,2026-05-12 23:59:59.999,5.792050e+08,2520381,112560.2947,2.577485e+08,0
2,2026-05-13,2274.97,2323.36,2260.51,2260.81,125797.3998,2026-05-13 23:59:59.999,2.888417e+08,1277106,62205.8189,1.428917e+08,0


## 10. Data Quality Checks
Check for missing values and gaps between consecutive candles.

In [11]:
quality_data = get_binance_data(symbol="BTCUSDT", interval="15m", start_time=to_ms("01.01.2020T00:00"), limit=1500)
df_quality = raw_to_dataframe(quality_data)

print("=== Null values per column ===")
print(df_quality.isnull().sum())

print("\n=== Gap check: close_time of row N vs open_time of row N+1 ===")
gaps = []
for i in range(len(df_quality) - 1):
    expected = df_quality["close_time"].iloc[i]
    actual = df_quality["open_time"].iloc[i + 1]
    diff = (actual - expected).total_seconds()
    if abs(diff) > 1:
        gaps.append((i, diff))

if gaps:
    print(f"Found {len(gaps)} gap(s): {gaps}")
else:
    print("No gaps found — candles are continuous.")

=== Null values per column ===
open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
ignore                    0
dtype: int64

=== Gap check: close_time of row N vs open_time of row N+1 ===
No gaps found — candles are continuous.


In [12]:
df_quality.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2020-01-01 00:00:00,7195.24,7196.25,7178.20,7180.97,202.942868,2020-01-01 00:14:59.999,1.458245e+06,2452,76.962041,553069.534929,0
1,2020-01-01 00:15:00,7180.97,7186.40,7175.47,7178.45,128.242654,2020-01-01 00:29:59.999,9.207027e+05,1948,58.389110,419226.471447,0
2,2020-01-01 00:30:00,7178.19,7185.44,7176.23,7179.56,83.487458,2020-01-01 00:44:59.999,5.994792e+05,1580,43.822374,314667.321072,0
3,2020-01-01 00:45:00,7179.35,7183.98,7175.46,7177.02,97.141921,2020-01-01 00:59:59.999,6.974298e+05,1660,46.979601,337325.862203,0
4,2020-01-01 01:00:00,7176.47,7194.04,7175.71,7190.86,103.520522,2020-01-01 01:14:59.999,7.440891e+05,1588,56.251378,404330.498386,0
